# 2025 Season Data Preparation

This notebook prepares and cleans 2025 MLB game data for modeling, including organizing at-bats, creating key features, and ensuring the data is ready for analysis.

In [1]:
!pip install pybaseball

In [2]:
"""# Import necessary modules
from google.colab import drive
import sys

# Mount Google Drive to access files stored in your Google Drive
drive.mount('/content/drive')

# NOTE: Update the paths below to match the location of your project files in Google Drive.
# Replace with your own directory if different.

# Add the main 'utils' directory to Python's module search path
# This allows you to import custom utility modules from this folder
path = "/content/drive/MyDrive/Homerun Prediction Model/utils"
sys.path.append(path)"""


'# Import necessary modules\nfrom google.colab import drive\nimport sys\n\n# Mount Google Drive to access files stored in your Google Drive\ndrive.mount(\'/content/drive\')\n\n# NOTE: Update the paths below to match the location of your project files in Google Drive.\n# Replace with your own directory if different.\n\n# Add the main \'utils\' directory to Python\'s module search path\n# This allows you to import custom utility modules from this folder\npath = "/content/drive/MyDrive/Homerun Prediction Model/utils"\nsys.path.append(path)'

In [3]:
from pathlib import Path
import sys

# Find the repo root by searching upwards for src/hrmodel
here = Path.cwd()
for p in (here, *here.parents):
    if (p / "src" / "hrmodel").exists():
        REPO = p
        break
else:
    raise RuntimeError("Could not find repo root containing src/hrmodel")

# Put src/ on sys.path (front so it wins)
sys.path.insert(0, str(REPO / "src"))


In [4]:
# Import from your package
from hrmodel import (
    add_batter_names,
    add_batter_full_name,
    prepare_barrels
)

# PyBaseball functions for Statcast data
from pybaseball import statcast
from pybaseball import playerid_reverse_lookup

# Core Python and data libraries
import numpy as np
import pandas as pd
from datetime import datetime
import os

### Retrieving Statcast Data for the 2025 MLB Season

This code below retrieves MLB Statcast data for the 2025 season. The start date is fixed as March 18, 2025 (`2025-03-18`), and the end date is dynamically set to the current date. It automatically updates to fetch the latest available data up until today.


In [5]:
# Set the start date as fixed for 2025-03-18
start_date = "2025-03-18"

# Get the current date as the end date
end_date = datetime.today().strftime('%Y-%m-%d')

# Retrieve the data
season_2025 = statcast(start_dt=start_date, end_dt=end_date)

This is a large query, it may take a moment to complete


/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  1%|▏                                          | 1/185 [00:00<00:24,  7.66it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime witho

/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicit

/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicit

 18%|███████▋                                  | 34/185 [00:06<00:30,  4.94it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
 21%|████████▋                                 | 38/185 [00:07<00:21,  6.78it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessin

...

/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
 71%|█████████████████████████████▎           | 132/185 [00:23<00:08,  5.95it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future

 77%|███████████████████████████████▍         | 142/185 [00:25<00:08,  5.02it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future

 84%|██████████████████████████████████▎      | 155/185 [00:27<00:04,  6.49it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future

/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
 90%|█████████████████████████████████████    | 167/185 [00:30<00:03,  5.24it/s]/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future

100%|█████████████████████████████████████████| 185/185 [00:37<00:00,  4.94it/s]
/Users/samuelgartenstein/anaconda3/lib/python3.11/site-packages/pybaseball/statcast.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_data = pd.concat(dataframe_list, axis=0).convert_dtypes(convert_string=False)


In [6]:
# Displaying first ten rows
season_2025.head(10)

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
4087,SL,2025-09-17,89.7,-1.78,5.87,"Kelly, Michael",686765,547184,fielders_choice,hit_into_play,...,<NA>,2.45,-0.5,0.5,<NA>,-0.292018,8.786186,35.573476,33.120324,32.761022
4178,ST,2025-09-17,83.4,-1.82,5.79,"Kelly, Michael",686765,547184,NaN,foul,...,<NA>,3.4,-1.39,1.39,<NA>,-17.871105,42.443985,40.872316,25.85424,20.017319
4324,FC,2025-09-17,91.4,-1.74,5.94,"Kelly, Michael",686765,547184,NaN,ball,...,<NA>,2.18,-0.15,0.15,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4466,SI,2025-09-17,96.0,-1.71,5.87,"Kelly, Michael",665966,547184,sac_bunt,hit_into_play,...,<NA>,1.76,1.31,1.31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4153,FF,2025-09-17,96.6,1.7,5.49,"Murphy, Chris",701762,669684,strikeout,foul_tip,...,<NA>,0.93,0.54,0.54,<NA>,19.887287,-2.012768,38.091679,30.374457,24.111675
4251,SI,2025-09-17,93.9,2.01,5.36,"Murphy, Chris",701762,669684,NaN,foul,...,<NA>,1.74,1.48,1.48,<NA>,6.658973,12.471468,45.439575,28.574871,19.395445
4404,SI,2025-09-17,92.9,2.22,5.21,"Murphy, Chris",701762,669684,NaN,swinging_strike,...,<NA>,1.92,1.45,1.45,<NA>,18.918324,-4.325163,46.029205,22.580741,28.514948
4538,SI,2025-09-17,93.2,2.12,5.19,"Murphy, Chris",701762,669684,NaN,ball,...,<NA>,1.98,1.23,1.23,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3596,SI,2025-09-17,96.8,-2.9,5.66,"Kelly, Zack",680869,677161,field_out,hit_into_play,...,<NA>,1.33,1.45,1.45,<NA>,15.336967,3.056715,34.885917,26.146564,22.406657
3732,FF,2025-09-17,97.3,-2.87,5.72,"Kelly, Zack",687231,677161,strikeout,swinging_strike,...,<NA>,1.14,0.86,0.86,<NA>,-11.835547,33.99038,34.218113,27.987509,10.932743


## Examining Data

### Columns

In [7]:
season_2025.columns.tolist()

['pitch_type',
 'game_date',
 'release_speed',
 'release_pos_x',
 'release_pos_z',
 'player_name',
 'batter',
 'pitcher',
 'events',
 'description',
 'spin_dir',
 'spin_rate_deprecated',
 'break_angle_deprecated',
 'break_length_deprecated',
 'zone',
 'des',
 'game_type',
 'stand',
 'p_throws',
 'home_team',
 'away_team',
 'type',
 'hit_location',
 'bb_type',
 'balls',
 'strikes',
 'game_year',
 'pfx_x',
 'pfx_z',
 'plate_x',
 'plate_z',
 'on_3b',
 'on_2b',
 'on_1b',
 'outs_when_up',
 'inning',
 'inning_topbot',
 'hc_x',
 'hc_y',
 'tfs_deprecated',
 'tfs_zulu_deprecated',
 'umpire',
 'sv_id',
 'vx0',
 'vy0',
 'vz0',
 'ax',
 'ay',
 'az',
 'sz_top',
 'sz_bot',
 'hit_distance_sc',
 'launch_speed',
 'launch_angle',
 'effective_speed',
 'release_spin_rate',
 'release_extension',
 'game_pk',
 'fielder_2',
 'fielder_3',
 'fielder_4',
 'fielder_5',
 'fielder_6',
 'fielder_7',
 'fielder_8',
 'fielder_9',
 'release_pos_y',
 'estimated_ba_using_speedangle',
 'estimated_woba_using_speedangle',
 'w

### Exploring Unique Events and Launch Speed Angles

This code prints the unique values for two key features in the `season_2025` dataset:

- **Events**: Lists the unique game events (e.g., hits, walks, strikeouts) that describe the outcome of each at-bat. It's important to check if "homerun" is included in the list of events.
  
- **Launch Speed Angle**: Displays the unique values for the angle at which the ball leaves the bat, providing insights into the trajectory of the ball. A launch speed angle of 6 corresponds to a "barrel," which is a well-hit ball.

In [8]:
print(season_2025['events'].unique(), "\n")
print(season_2025['launch_speed_angle'].unique(), "\n")

['fielders_choice' nan 'sac_bunt' 'strikeout' 'field_out' 'single' 'walk'
 'truncated_pa' 'field_error' 'force_out' 'grounded_into_double_play'
 'home_run' 'sac_fly' 'double' 'triple' 'catcher_interf' 'double_play'
 'hit_by_pitch' 'fielders_choice_out' 'strikeout_double_play'
 'sac_fly_double_play' 'triple_play'] 

<IntegerArray>
[1, <NA>, 3, 4, 6, 2, 5]
Length: 7, dtype: Int64 



#### Changing `player_name` to `pitcher_name` and `batter` to `batter_id`


In [9]:
season_2025 = season_2025.rename(columns={
    "player_name": "pitcher_name",
    "batter": "batter_id"
})

## Mapping Batter IDs to Player Names

Although not necessary for running the model, I will match the `batter` ID to the player's name. This will allow us to predict individual batter homerun probabilities. This can be done using the `playerid_reverse_lookup` function, which returns a DataFrame containing the player ID (`key_mlbam`) along with the batter's first and last name.


In [10]:
# Add batter names to the 2025 season dataset
season_2025 = add_batter_names(season_2025)
season_2025.head(10)

Gathering player lookup table. This may take a moment.


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,SL,2025-09-17,89.7,-1.78,5.87,"Kelly, Michael",686765,547184,fielders_choice,hit_into_play,...,33.120324,32.761022,sogard,nick,686765.0,sogan001,sogarni01,-1.0,2024.0,2025.0
1,ST,2025-09-17,83.4,-1.82,5.79,"Kelly, Michael",686765,547184,NaN,foul,...,25.85424,20.017319,sogard,nick,686765.0,sogan001,sogarni01,-1.0,2024.0,2025.0
2,FC,2025-09-17,91.4,-1.74,5.94,"Kelly, Michael",686765,547184,NaN,ball,...,<NA>,<NA>,sogard,nick,686765.0,sogan001,sogarni01,-1.0,2024.0,2025.0
3,SI,2025-09-17,96.0,-1.71,5.87,"Kelly, Michael",665966,547184,sac_bunt,hit_into_play,...,<NA>,<NA>,narváez,carlos,665966.0,narvc002,narvaca01,-1.0,2024.0,2025.0
4,FF,2025-09-17,96.6,1.7,5.49,"Murphy, Chris",701762,669684,strikeout,foul_tip,...,30.374457,24.111675,kurtz,nick,701762.0,NaN,kurtzni01,-1.0,2025.0,2025.0
5,SI,2025-09-17,93.9,2.01,5.36,"Murphy, Chris",701762,669684,NaN,foul,...,28.574871,19.395445,kurtz,nick,701762.0,NaN,kurtzni01,-1.0,2025.0,2025.0
6,SI,2025-09-17,92.9,2.22,5.21,"Murphy, Chris",701762,669684,NaN,swinging_strike,...,22.580741,28.514948,kurtz,nick,701762.0,NaN,kurtzni01,-1.0,2025.0,2025.0
7,SI,2025-09-17,93.2,2.12,5.19,"Murphy, Chris",701762,669684,NaN,ball,...,<NA>,<NA>,kurtz,nick,701762.0,NaN,kurtzni01,-1.0,2025.0,2025.0
8,SI,2025-09-17,96.8,-2.9,5.66,"Kelly, Zack",680869,677161,field_out,hit_into_play,...,26.146564,22.406657,gelof,zack,680869.0,geloz001,gelofza01,29766.0,2023.0,2025.0
9,FF,2025-09-17,97.3,-2.87,5.72,"Kelly, Zack",687231,677161,strikeout,swinging_strike,...,27.987509,10.932743,hernaiz,darell,687231.0,hernd006,hernada04,26224.0,2024.0,2025.0


### Creating a Full Name Column for Batters

To streamline player identification in the dataset, this step standardizes the formatting of first and last names by capitalizing them and then concatenates them into a new `full_name` column.

In [11]:
season_2025 = add_batter_full_name(season_2025)
season_2025

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,SL,2025-09-17,89.7,-1.78,5.87,"Kelly, Michael",686765,547184,fielders_choice,hit_into_play,...,32.761022,Sogard,Nick,686765.0,sogan001,sogarni01,-1.0,2024.0,2025.0,Nick Sogard
1,ST,2025-09-17,83.4,-1.82,5.79,"Kelly, Michael",686765,547184,NaN,foul,...,20.017319,Sogard,Nick,686765.0,sogan001,sogarni01,-1.0,2024.0,2025.0,Nick Sogard
2,FC,2025-09-17,91.4,-1.74,5.94,"Kelly, Michael",686765,547184,NaN,ball,...,<NA>,Sogard,Nick,686765.0,sogan001,sogarni01,-1.0,2024.0,2025.0,Nick Sogard
3,SI,2025-09-17,96.0,-1.71,5.87,"Kelly, Michael",665966,547184,sac_bunt,hit_into_play,...,<NA>,Narváez,Carlos,665966.0,narvc002,narvaca01,-1.0,2024.0,2025.0,Carlos Narváez
4,FF,2025-09-17,96.6,1.7,5.49,"Murphy, Chris",701762,669684,strikeout,foul_tip,...,24.111675,Kurtz,Nick,701762.0,NaN,kurtzni01,-1.0,2025.0,2025.0,Nick Kurtz
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697260,FS,2025-03-18,83.1,2.67,5.42,"Imanaga, Shota",669242,684007,NaN,swinging_strike,...,45.254212,Edman,Tommy,669242.0,edmat001,edmanto01,19470.0,2019.0,2025.0,Tommy Edman
697261,FS,2025-03-18,83.2,2.79,5.49,"Imanaga, Shota",669242,684007,NaN,ball,...,<NA>,Edman,Tommy,669242.0,edmat001,edmanto01,19470.0,2019.0,2025.0,Tommy Edman
697262,FF,2025-03-18,93.1,2.45,5.46,"Imanaga, Shota",660271,684007,field_out,hit_into_play,...,37.667497,Ohtani,Shohei,660271.0,ohtas001,ohtansh01,19755.0,2018.0,2025.0,Shohei Ohtani
697263,SL,2025-03-18,83.1,2.88,5.31,"Imanaga, Shota",660271,684007,NaN,ball,...,<NA>,Ohtani,Shohei,660271.0,ohtas001,ohtansh01,19755.0,2018.0,2025.0,Shohei Ohtani


### Keeping Only Necessary Columns

To simplify the dataset and focus on the most relevant features for analysis, this step selects a subset of columns from the full `season_2025` DataFrame. These columns include game context (e.g., inning, teams), player identifiers, and pitch outcome details.

In [12]:
# Keep only the relevant columns for analysis
season_2025 = season_2025[
    ['game_date',           # Date of the game
     'pitcher_name',        # Name of the pitcher
     'home_team',           # Home team
     'away_team',           # Away team
     'inning',              # Inning number
     'inning_topbot',       # Top or bottom half of the inning
     'at_bat_number',       # Number of the at-bat within the game
     'batter_name',           # Full name of the batter
     'batter_id',              # Batter ID
     'pitch_number',        # Pitch number within the at-bat
     'outs_when_up',        # Number of outs when the batter was up
     'p_throws',            # Pitcher's throwing hand (L or R)
     'events',              # Resulting event of the play (e.g., single, strikeout)
     'description',         # Description of the pitch (e.g., called strike, swinging strike)
     'launch_speed_angle'   # Angle at which the ball leaves the bat after contact (trajectory insight)
     ]
]

season_2025.head(10)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle
0,2025-09-17,"Kelly, Michael",BOS,ATH,10,Bot,92,Nick Sogard,686765,3,1,R,fielders_choice,hit_into_play,1
1,2025-09-17,"Kelly, Michael",BOS,ATH,10,Bot,92,Nick Sogard,686765,2,1,R,NaN,foul,<NA>
2,2025-09-17,"Kelly, Michael",BOS,ATH,10,Bot,92,Nick Sogard,686765,1,1,R,NaN,ball,<NA>
3,2025-09-17,"Kelly, Michael",BOS,ATH,10,Bot,91,Carlos Narváez,665966,1,0,R,sac_bunt,hit_into_play,1
4,2025-09-17,"Murphy, Chris",BOS,ATH,10,Top,89,Nick Kurtz,701762,4,2,L,strikeout,foul_tip,<NA>
5,2025-09-17,"Murphy, Chris",BOS,ATH,10,Top,89,Nick Kurtz,701762,3,2,L,NaN,foul,<NA>
6,2025-09-17,"Murphy, Chris",BOS,ATH,10,Top,89,Nick Kurtz,701762,2,2,L,NaN,swinging_strike,<NA>
7,2025-09-17,"Murphy, Chris",BOS,ATH,10,Top,89,Nick Kurtz,701762,1,2,L,NaN,ball,<NA>
8,2025-09-17,"Kelly, Zack",BOS,ATH,10,Top,88,Zack Gelof,680869,1,1,R,field_out,hit_into_play,3
9,2025-09-17,"Kelly, Zack",BOS,ATH,10,Top,87,Darell Hernaiz,687231,4,0,R,strikeout,swinging_strike,<NA>


## Starting Pitcher

Below, an indicator will be assigned to each starting pitcher. This indicator will be used as a predictor in one of the models to evaluate the impact of the starting pitcher on the game.


In [13]:
# Sort the DataFrame by multiple columns for proper ordering
season_2025 = season_2025.sort_values(
    by=['pitcher_name', 'game_date', 'inning', 'at_bat_number', 'pitch_number', 'outs_when_up']  # Sorting order
).reset_index(drop=True)  # Reset index after sorting

# Identify the starting pitcher for each game
starter_ids = (
    season_2025[
        (season_2025['inning'] == 1) &  # First inning
        (season_2025['pitch_number'] == 1)  # First pitch of the game
    ][['game_date', 'pitcher_name']]  # Select relevant columns (game_date and pitcher_name)
    .drop_duplicates()  # Remove duplicate entries for each game-pitcher combination
    .assign(starting_pitcher=1)  # Assign a value of 1 to indicate the starting pitcher
)

# Merge the starting pitcher information with the main dataset
season_2025 = season_2025.merge(
    starter_ids,  # Merge using the starting pitcher data
    on=['game_date', 'pitcher_name'],  # Match on game_date and pitcher_name
    how='left'  # Perform a left join to retain all rows from season_2025
)

# Fill missing values in 'starting_pitcher' column and convert to integer type
season_2025['starting_pitcher'] = season_2025['starting_pitcher'].fillna(0).astype(int)

# Display the first 10 rows of the updated DataFrame
season_2025.head(10)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,0,L,field_out,hit_into_play,3,1
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,1,1,L,NaN,called_strike,<NA>,1
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,2,1,L,NaN,foul,<NA>,1
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,3,1,L,NaN,ball,<NA>,1
4,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,1,L,field_out,hit_into_play,1,1
5,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,1,2,L,NaN,ball,<NA>,1
6,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,2,2,L,NaN,foul,<NA>,1
7,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,2,L,single,hit_into_play,2,1
8,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,1,2,L,NaN,ball,<NA>,1
9,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,2,2,L,NaN,blocked_ball,<NA>,1


## Barrels



Now we can calculate barrels, defined here as cases where `launch_speed_angle` equals `6`. Before doing so, we first filter the dataset to keep only the final pitch of each at-bat, since the at-bat outcome (such as a home run) is determined on that pitch. This makes the at-bat our unit of observation and ensures we are focusing on the decisive moment. Finally, we create binary indicators to capture whether the last pitch was a barrel and whether it resulted in a home run.





In [14]:
at_bats_2025 = prepare_barrels(season_2025)
at_bats_2025.head(10)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel
91314,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,called_strike,<NA>,1,0
91346,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,1,R,double,hit_into_play,6,1,1
91373,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,2,R,force_out,hit_into_play,2,1,0
548865,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,swinging_strike,<NA>,1,0
548901,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,0,R,force_out,hit_into_play,2,1,0
604393,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,1,Top,2,Aaron Judge,592450,6,1,R,walk,ball,<NA>,1,0
604429,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,3,Top,19,Aaron Judge,592450,4,0,R,walk,ball,<NA>,1,0
604463,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,5,Top,40,Aaron Judge,592450,8,0,R,strikeout,swinging_strike,<NA>,1,0
366594,2025-03-22,"Luzardo, Jesús",NYY,PHI,1,Bot,6,Aaron Judge,592450,6,1,L,strikeout,swinging_strike,<NA>,1,0
366623,2025-03-22,"Luzardo, Jesús",NYY,PHI,3,Bot,22,Aaron Judge,592450,3,2,L,home_run,hit_into_play,6,1,1


### Barrel Rate for Batters

The code below calculates each batter’s rolling barrel rate over their last 40 at-bats. This rolling average, which excludes the current at-bat, will serve as one of the predictors in the Bayesian model.

In [15]:
# Calculate each batter's rolling barrel rate over their last 40 at-bats.
# For each batter_id, the 'barrel' column is shifted by 1 (to exclude the current at-bat),
# then a rolling mean over the previous 40 at-bats is computed.
# This creates a "rolling_batter_barrel_rate" feature that tracks recent performance trends.
at_bats_2025["rolling_batter_barrel_rate"] = (
    at_bats_2025.groupby("batter_id")["barrel"]
    .transform(lambda x: x.shift(1).rolling(40, min_periods=1).mean())
)

at_bats_2025.head(20)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel,rolling_batter_barrel_rate
91314,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,called_strike,<NA>,1,0,NaN
91346,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,1,R,double,hit_into_play,6,1,1,0.000000
91373,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,2,R,force_out,hit_into_play,2,1,0,0.500000
548865,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,swinging_strike,<NA>,1,0,0.333333
548901,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,0,R,force_out,hit_into_play,2,1,0,0.250000
604393,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,1,Top,2,Aaron Judge,592450,6,1,R,walk,ball,<NA>,1,0,0.200000
604429,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,3,Top,19,Aaron Judge,592450,4,0,R,walk,ball,<NA>,1,0,0.166667
604463,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,5,Top,40,Aaron Judge,592450,8,0,R,strikeout,swinging_strike,<NA>,1,0,0.142857
366594,2025-03-22,"Luzardo, Jesús",NYY,PHI,1,Bot,6,Aaron Judge,592450,6,1,L,strikeout,swinging_strike,<NA>,1,0,0.125000
366623,2025-03-22,"Luzardo, Jesús",NYY,PHI,3,Bot,22,Aaron Judge,592450,3,2,L,home_run,hit_into_play,6,1,1,0.111111


### Barrel Rate for Pitchers

The code below calculates each pitcher's rolling barrel rate (given up) over their last 40 at-bats. Just like for batters, this excludes the current at-bat and will serve as one of the predictors in the Bayesian model.

In [16]:
# Sort pitches by pitcher and game context to ensure proper sequence
at_bats_2025 = at_bats_2025.sort_values(
    by=['pitcher_name', 'game_date', 'inning', 'at_bat_number', 'pitch_number', 'outs_when_up']
).reset_index(drop=True)

# Calculate each pitcher's rolling barrel rate over their last 40 at-bats faced.
# For each pitcher_name, the 'barrel' column is shifted by 1 (to exclude the current at-bat),
# then a rolling mean is taken over the previous 40 at-bats.
# This "rolling_pitcher_barrel_rate" feature tracks how often batters barrel pitches
# against the pitcher in their recent appearances.
at_bats_2025["rolling_pitcher_barrel_rate"] = (
    at_bats_2025.groupby("pitcher_name")["barrel"]
    .transform(lambda x: x.shift(1).rolling(40, min_periods=1).mean())
)

at_bats_2025.head(20)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel,rolling_batter_barrel_rate,rolling_pitcher_barrel_rate
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,0,L,field_out,hit_into_play,3,1,0,0.000000,NaN
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,1,L,field_out,hit_into_play,1,1,0,0.000000,0.0
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,2,L,single,hit_into_play,2,1,0,0.000000,0.0
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,2,L,strikeout,swinging_strike,<NA>,1,0,0.000000,0.0
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,0,L,field_out,hit_into_play,4,1,0,0.000000,0.0
5,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,12,Jake Cronenworth,630105,5,1,L,strikeout,swinging_strike,<NA>,1,0,0.000000,0.0
6,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,13,Yuli Gurriel,493329,5,2,L,strikeout,swinging_strike,<NA>,1,0,0.000000,0.0
7,2025-03-22,"Abbott, Andrew",SD,CIN,3,Bot,17,Brandon Lockridge,663604,5,0,L,field_out,hit_into_play,3,1,0,0.000000,0.0
8,2025-03-22,"Abbott, Andrew",SD,CIN,3,Bot,18,Martín Maldonado,455117,2,1,L,field_out,hit_into_play,4,1,0,0.000000,0.0
9,2025-03-22,"Abbott, Andrew",SD,CIN,3,Bot,19,Fernando Tatís,665487,3,2,L,field_out,hit_into_play,2,1,0,0.000000,0.0


## Imputing Missing Data

First, when calculating rolling barrel rates for batters and pitchers, the initial game for each player will not have a prior value. These cases are imputed using the player’s barrel rate from the previous season. Importantly, this imputed value is used only for prediction purposes and is not included in the rolling calculation itself.

Second, if a player did not appear in the 2024 season, no prior-season data are available. In these cases, the missing barrel rate is imputed using the overall average barrel rate for all batters or pitchers, respectively.

### Reading 2024 Batter and Pitcher Barrel Data

The code below imports the CSV files containing the 2024 barrel rates for both batters and pitchers.

In [17]:
batter_path = "data/batter_barrel_means_2024.csv"
pitcher_path = "data/pitcher_barrel_means_2024.csv"

In [18]:
batter_barrel_means_2024 = pd.read_csv(batter_path)
batter_barrel_means_2024.head(10)

,batter_id,batter_mean_barrel_rate_2024
0,444482,0.036765
1,453568,0.040936
2,455117,0.044872
3,456781,0.025890
4,457705,0.072106
5,457759,0.034483
6,462101,0.000000
7,467055,0.000000
8,467793,0.051240
9,493329,0.015385


In [19]:
pitcher_barrel_means_2024 = pd.read_csv(pitcher_path)
pitcher_barrel_means_2024.head(10)

,pitcher_name,pitcher_mean_barrel_rate_allowed_2024
0,"Abbott, Andrew",0.066553
1,"Abney, Alaska",0.000000
2,"Abreu, Bryan",0.048632
3,"Adam, Jason",0.037543
4,"Adams, Austin",0.030612
5,"Adcock, Ty",0.190476
6,"Adon, Joan",0.062500
7,"Adón, Melvin",0.000000
8,"Agnos, Zach",0.000000
9,"Aguiar, Julian",0.085714


### Imputing Rolling Rates

This function fills in missing rolling rate values by using the group’s average from the previous season (`means_df`).  
It can also optionally sort the data first (by player and game context) to ensure the order is correct before imputation.  

In [20]:
def impute_rolling_rates(
    df: pd.DataFrame,
    rolling_col: str,
    group_col: str,
    means_df: pd.DataFrame,
    mean_col: str,
    sort_first_by: str | None = None,
) -> pd.DataFrame:
    """
    Impute missing rolling rates using previous-season group means.
    If no group mean is available, fall back to the overall mean.
    Optionally sort by a chosen column (e.g., 'batter_name' or 'pitcher_name')
    plus standard game-context keys to ensure correct ordering before imputation.
    """
    if sort_first_by is not None:
        # Only include columns that exist to avoid KeyError
        ctx = [c for c in ['game_date', 'inning', 'at_bat_number', 'pitch_number', 'outs_when_up'] if c in df.columns]
        sort_cols = [sort_first_by] + ctx
        df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    # Merge group means onto current season data
    df = df.merge(means_df, how="left", on=group_col)

    # Calculate overall mean (true mean of rolling_col, ignoring NaNs)
    overall_mean = df[rolling_col].mean(skipna=True)

    # Fill missing rolling rates: first with group mean, then with overall mean
    df[rolling_col] = df[rolling_col].fillna(df[mean_col]).fillna(overall_mean)

    # Drop the temporary mean column
    df = df.drop(columns=[mean_col])

    return df

#### Batter Barrel Imputation

In [21]:
at_bats_2025 = impute_rolling_rates(
    at_bats_2025,
    rolling_col="rolling_batter_barrel_rate",
    group_col="batter_id",
    means_df=batter_barrel_means_2024,
    mean_col="batter_mean_barrel_rate_2024",
    sort_first_by="batter_name",
)

at_bats_2025.head(10)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel,rolling_batter_barrel_rate,rolling_pitcher_barrel_rate
0,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,called_strike,<NA>,1,0,0.152080,0.000000
1,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,1,R,double,hit_into_play,6,1,1,0.000000,0.200000
2,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,2,R,force_out,hit_into_play,2,1,0,0.500000,0.210526
3,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,swinging_strike,<NA>,1,0,0.333333,0.000000
4,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,0,R,force_out,hit_into_play,2,1,0,0.250000,0.000000
5,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,1,Top,2,Aaron Judge,592450,6,1,R,walk,ball,<NA>,1,0,0.200000,0.000000
6,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,3,Top,19,Aaron Judge,592450,4,0,R,walk,ball,<NA>,1,0,0.166667,0.000000
7,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,5,Top,40,Aaron Judge,592450,8,0,R,strikeout,swinging_strike,<NA>,1,0,0.142857,0.000000
8,2025-03-22,"Luzardo, Jesús",NYY,PHI,1,Bot,6,Aaron Judge,592450,6,1,L,strikeout,swinging_strike,<NA>,1,0,0.125000,0.000000
9,2025-03-22,"Luzardo, Jesús",NYY,PHI,3,Bot,22,Aaron Judge,592450,3,2,L,home_run,hit_into_play,6,1,1,0.111111,0.100000


#### Pitcher Barrel Imputation

In [22]:
at_bats_2025 = impute_rolling_rates(
    at_bats_2025,
    rolling_col="rolling_pitcher_barrel_rate",
    group_col="pitcher_name",
    means_df=pitcher_barrel_means_2024,
    mean_col="pitcher_mean_barrel_rate_allowed_2024",
    sort_first_by="pitcher_name",
)

at_bats_2025

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel,rolling_batter_barrel_rate,rolling_pitcher_barrel_rate
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,0,L,field_out,hit_into_play,3,1,0,0.00,0.066553
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,1,L,field_out,hit_into_play,1,1,0,0.00,0.000000
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,2,L,single,hit_into_play,2,1,0,0.00,0.000000
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,2,L,strikeout,swinging_strike,<NA>,1,0,0.00,0.000000
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,0,L,field_out,hit_into_play,4,1,0,0.00,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178245,2025-09-17,"deGrom, Jacob",HOU,TEX,4,Bot,35,Zach Cole,805904,3,2,R,strikeout,swinging_strike,<NA>,1,0,0.10,0.150000
178246,2025-09-17,"deGrom, Jacob",HOU,TEX,5,Bot,39,Jeremy Peña,665161,5,0,R,home_run,hit_into_play,6,1,1,0.05,0.150000
178247,2025-09-17,"deGrom, Jacob",HOU,TEX,5,Bot,40,Carlos Correa,621043,8,0,R,field_out,hit_into_play,4,1,0,0.05,0.175000
178248,2025-09-17,"deGrom, Jacob",HOU,TEX,5,Bot,41,José Altuve,514888,2,1,R,field_out,hit_into_play,3,1,0,0.05,0.175000


### Checking For Missing Values



In [23]:
# Check for missing values in each column
print("Missing in rolling_batter_barrel_rate:", at_bats_2025['rolling_batter_barrel_rate'].isna().sum())
print("Missing in rolling_pitcher_barrel_rate:", at_bats_2025['rolling_pitcher_barrel_rate'].isna().sum())


Missing in rolling_batter_barrel_rate: 0
Missing in rolling_pitcher_barrel_rate: 0


## Homerun

An indicator is created for the binary response variable showing whether or not a player hit a home run.

In [24]:
# Sort the DataFrame to ensure at-bats are ordered properly for each batter
at_bats_2025 = at_bats_2025.sort_values(
    by=["batter_name", "game_date", "inning", "at_bat_number"]
)

# Create a binary indicator for home runs
# 'home_run' = 1 if the final pitch event was a home run, otherwise 0
at_bats_2025["home_run"] = (at_bats_2025["events"] == "home_run").astype(int)

at_bats_2025.head(20)


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel,rolling_batter_barrel_rate,rolling_pitcher_barrel_rate,home_run
23244,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,called_strike,<NA>,1,0,0.152080,0.000000,0
23253,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,1,R,double,hit_into_play,6,1,1,0.000000,0.200000,0
23262,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,2,R,force_out,hit_into_play,2,1,0,0.500000,0.210526,0
140283,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,swinging_strike,<NA>,1,0,0.333333,0.000000,0
140292,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,0,R,force_out,hit_into_play,2,1,0,0.250000,0.000000,0
154472,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,1,Top,2,Aaron Judge,592450,6,1,R,walk,ball,<NA>,1,0,0.200000,0.000000,0
154480,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,3,Top,19,Aaron Judge,592450,4,0,R,walk,ball,<NA>,1,0,0.166667,0.000000,0
154488,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,5,Top,40,Aaron Judge,592450,8,0,R,strikeout,swinging_strike,<NA>,1,0,0.142857,0.000000,0
93902,2025-03-22,"Luzardo, Jesús",NYY,PHI,1,Bot,6,Aaron Judge,592450,6,1,L,strikeout,swinging_strike,<NA>,1,0,0.125000,0.000000,0
93911,2025-03-22,"Luzardo, Jesús",NYY,PHI,3,Bot,22,Aaron Judge,592450,3,2,L,home_run,hit_into_play,6,1,1,0.111111,0.100000,1


### Homerun Previous Game

Another predictor is whether a player hit a home run in their previous game. This feature is intended to capture potential “hot streaks,” where recent performance may influence the likelihood of hitting another home run.

In [25]:
# Create a flag if the batter hit a home run in the current game
# First, get the total HRs per batter per game
batter_game_hr = (
    at_bats_2025.groupby(["batter_name", "game_date"])["home_run"]
    .sum()
    .reset_index()
    .assign(hit_hr=lambda df: (df["home_run"] > 0).astype(int))
)

# Shift the flag to get the previous game's HR indicator
batter_game_hr["prev_game_hr"] = (
    batter_game_hr.groupby("batter_name")["hit_hr"].shift().fillna(0).astype(int)
)

# Merge back to the original at-bat level DataFrame
at_bats_2025 = at_bats_2025.merge(
    batter_game_hr[["batter_name", "game_date", "prev_game_hr"]],
    on=["batter_name", "game_date"],
    how="left"
)

at_bats_2025.head(20)

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,p_throws,events,description,launch_speed_angle,starting_pitcher,barrel,rolling_batter_barrel_rate,rolling_pitcher_barrel_rate,home_run,prev_game_hr
0,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,...,R,strikeout,called_strike,<NA>,1,0,0.152080,0.000000,0,0
1,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,...,R,double,hit_into_play,6,1,1,0.000000,0.200000,0,0
2,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,...,R,force_out,hit_into_play,2,1,0,0.500000,0.210526,0,0
3,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,...,R,strikeout,swinging_strike,<NA>,1,0,0.333333,0.000000,0,0
4,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,...,R,force_out,hit_into_play,2,1,0,0.250000,0.000000,0,0
5,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,1,Top,2,Aaron Judge,592450,6,...,R,walk,ball,<NA>,1,0,0.200000,0.000000,0,0
6,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,3,Top,19,Aaron Judge,592450,4,...,R,walk,ball,<NA>,1,0,0.166667,0.000000,0,0
7,2025-03-20,"Sugano, Tomoyuki",BAL,NYY,5,Top,40,Aaron Judge,592450,8,...,R,strikeout,swinging_strike,<NA>,1,0,0.142857,0.000000,0,0
8,2025-03-22,"Luzardo, Jesús",NYY,PHI,1,Bot,6,Aaron Judge,592450,6,...,L,strikeout,swinging_strike,<NA>,1,0,0.125000,0.000000,0,0
9,2025-03-22,"Luzardo, Jesús",NYY,PHI,3,Bot,22,Aaron Judge,592450,3,...,L,home_run,hit_into_play,6,1,1,0.111111,0.100000,1,0


## Additional Preprocessing

## Exporting 2025 At-Bats data

In [26]:
# Define full path to your project folder
drive_path = "data"

at_bats_2025.to_csv(os.path.join(drive_path, "at_bats_2025.csv"), index=False)